## 0. Cài đặt môi trường & Thư viện

In [7]:
import pandas as pd
import numpy as np
from scipy import stats

# Thiết lập cấu hình hiển thị bảng của Pandas
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Đã import thành công: pandas, numpy, scipy.stats!')

Đã import thành công: pandas, numpy, scipy.stats!


## 1. Đọc và Khám Phá Cấu Trúc Tập Dữ Liệu

In [8]:
csv_path = 'hanoi_restaurants_cleaned.csv'
df = pd.read_csv(csv_path)

print(f'1. Kích thước tập dữ liệu: {df.shape[0]} nhà hàng, {df.shape[1]} thuộc tính/cột.')
print('\n2. Danh sách các cột:')
for i, c in enumerate(df.columns, 1):
    print(f'   {i:02d}. {c} (kiểu: {df[c].dtype})')

print('\n3. 5 dòng dữ liệu đầu tiên:')
display(df.head())

1. Kích thước tập dữ liệu: 1210 nhà hàng, 17 thuộc tính/cột.

2. Danh sách các cột:
   01. Restaurant ID (kiểu: object)
   02. Restaurant Name (kiểu: object)
   03. Address (kiểu: object)
   04. Locality (kiểu: object)
   05. Locality Verbose (kiểu: object)
   06. Cuisines (kiểu: object)
   07. Average Cost for two (kiểu: int64)
   08. Currency (kiểu: object)
   09. Has Table booking (kiểu: object)
   10. Has Online delivery (kiểu: object)
   11. Is delivering now (kiểu: object)
   12. Price range (kiểu: int64)
   13. Aggregate rating (kiểu: float64)
   14. Rating text (kiểu: object)
   15. Votes (kiểu: int64)
   16. latitude (kiểu: float64)
   17. longitude (kiểu: float64)

3. 5 dòng dữ liệu đầu tiên:


,Restaurant ID,Restaurant Name,Address,Locality,Locality Verbose,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Price range,Aggregate rating,Rating text,Votes,latitude,longitude
0,HN0001,Steakhouse & Wine Bách Khoa - Cầu Giấy,"Số 92 Đường Cầu Giấy, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Đồ Âu, bít tết, rượu vang",576000,VND,Yes,Yes,Yes,4,4.50,Xuất sắc,727,21.03,105.79
1,HN0002,Bún Đậu Minh Khang - Duy Tân,"Số 191 Phố Duy Tân, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Việt, quán bún đậu",107000,VND,No,Yes,Yes,1,3.80,Tốt,468,21.03,105.80
2,HN0003,Pizza & Pasta Trung Hòa - Tô Hiệu,"Số 32 Phố Tô Hiệu, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Ý, Đồ Âu",167000,VND,No,Yes,Yes,2,3.90,Tốt,599,21.03,105.80
3,HN0004,Buffet Hải Sản & BBQ Linh Lang - Xuân Thủy,"Số 115 Đường Xuân Thủy, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Việt, hải sản, Buffet",562000,VND,Yes,Yes,Yes,4,4.30,Rất tốt,741,21.03,105.80
4,HN0005,Bún Riêu Cua Maison - Trung Hòa,"Số 218 Phố Trung Hòa, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Việt, quán bún riêu",74000,VND,No,Yes,Yes,1,3.70,Tốt,517,21.03,105.80


## 2. Kiểm Tra Chất Lượng Dữ Liệu (Missing Values & Duplicates)

In [9]:
# Kiểm tra giá trị khuyết thiếu (Missing values)
missing_check = pd.DataFrame({
    'Số lượng null': df.isnull().sum(),
    'Tỷ lệ null (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

print('Báo cáo giá trị thiếu từng cột:')
display(missing_check)

# Kiểm tra dòng trùng lặp
dup_count = df.duplicated(subset=['Restaurant ID']).sum()
print(f'Số lượng Restaurant ID trùng lặp: {dup_count}')

Báo cáo giá trị thiếu từng cột:


,Số lượng null,Tỷ lệ null (%)
Restaurant ID,0,0.00
Restaurant Name,0,0.00
Address,0,0.00
Locality,0,0.00
Locality Verbose,0,0.00
Cuisines,0,0.00
Average Cost for two,0,0.00
Currency,0,0.00
Has Table booking,0,0.00
Has Online delivery,0,0.00


Số lượng Restaurant ID trùng lặp: 0


## 3. Thống Kê Mô Tả Chi Tiết Các Biến Định Lượng
Thay thế cho **Histogram** và **Boxplot**, chúng ta sử dụng:
- Thống kê phân vị chuyên sâu (5%, 10%, 25%, 50%, 75%, 90%, 95%, 99%).
- Độ xiên (Skewness) & Độ nhọn (Kurtosis) để xác định độ lệch phân phối.
- Quy tắc $1.5 \times \text{IQR}$ để phát hiện các nhà hàng có giá hoặc số votes ngoại lai.

In [10]:
num_vars = ['Average Cost for two', 'Aggregate rating', 'Votes', 'Price range']

# 1. Bảng tóm tắt thống kê đa phân vị
desc_stats = df[num_vars].describe(
    percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
print('--- Thống kê mô tả & Phân vị chuyên sâu ---')
display(desc_stats)

# 2. Đo lường hình dạng phân phối: Skewness & Kurtosis
# Skew > 0: Phân phối lệch phải (phần lớn giá trị nhỏ, có một số quán giá cực cao)
# Kurtosis > 3 (hoặc fisher > 0): Đuôi dày hơn phân phối chuẩn, có nhiều điểm cực trị
shape_analysis = pd.DataFrame({
    'Độ xiên (Skewness)': df[num_vars].skew(),
    'Độ nhọn (Kurtosis)': df[num_vars].kurtosis()
})
print('\n--- Hình dạng phân phối (Độ xiên & Độ nhọn) ---')
display(shape_analysis)

# 3. Phát hiện Outliers bằng phương pháp IQR (Thay thế cho Boxplot)
def get_outliers_summary(series, col_name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers = series[(series < low) | (series > high)]
    return {
        'Cột': col_name,
        'Q1 (25%)': q1,
        'Q3 (75%)': q3,
        'IQR': iqr,
        'Ngưỡng dưới': max(0, low),
        'Ngưỡng trên': high,
        'Số Outliers': len(outliers),
        'Tỷ lệ Outliers (%)': round(len(outliers) / len(series) * 100, 2)
    }

outliers_report = pd.DataFrame([
    get_outliers_summary(df['Average Cost for two'], 'Average Cost for two'),
    get_outliers_summary(df['Aggregate rating'], 'Aggregate rating'),
    get_outliers_summary(df['Votes'], 'Votes')
]).set_index('Cột')

print('\n--- Báo cáo Outliers theo phương pháp IQR ---')
display(outliers_report)

# Xem thử một số quán ăn có chi phí ngoại lai cao nhất
top_cost_outliers = df.sort_values(by='Average Cost for two', ascending=False)[[
    'Restaurant Name', 'Locality', 'Cuisines', 'Average Cost for two', 'Aggregate rating'
]].head(5)
print('\nTop 5 nhà hàng có chi phí đắt đỏ nhất:')
display(top_cost_outliers)

--- Thống kê mô tả & Phân vị chuyên sâu ---


,Average Cost for two,Aggregate rating,Votes,Price range
count,"1,210.00","1,210.00","1,210.00","1,210.00"
mean,"257,263.64",4.14,581.30,2.17
std,"185,028.34",0.32,363.54,1.05
min,"50,000.00",3.30,65.00,1.00
5%,"72,450.00",3.65,203.00,1.00
10%,"80,000.00",3.70,255.80,1.00
25%,"100,000.00",3.90,368.25,1.00
50%,"210,000.00",4.10,508.00,2.00
75%,"370,000.00",4.40,679.00,3.00
90%,"528,100.00",4.60,906.20,4.00



--- Hình dạng phân phối (Độ xiên & Độ nhọn) ---


,Độ xiên (Skewness),Độ nhọn (Kurtosis)
Average Cost for two,1.13,0.78
Aggregate rating,0.10,-0.49
Votes,2.97,14.35
Price range,0.31,-1.19



--- Báo cáo Outliers theo phương pháp IQR ---


,Q1 (25%),Q3 (75%),IQR,Ngưỡng dưới,Ngưỡng trên,Số Outliers,Tỷ lệ Outliers (%)
Cột,,,,,,,
Average Cost for two,"100,000.00","370,000.00","270,000.00",0.00,"775,000.00",18,1.49
Aggregate rating,3.90,4.40,0.50,3.15,5.15,0,0.00
Votes,368.25,679.00,310.75,0.00,"1,145.12",74,6.12



Top 5 nhà hàng có chi phí đắt đỏ nhất:


,Restaurant Name,Locality,Cuisines,Average Cost for two,Aggregate rating
639,Haidilao Hotpot Phạm Ngọc Thạch,Đống Đa,"Lẩu Trung Hoa, hải sản",950000,4.90
1018,Hatoyama Japanese Restaurant Tuấn - Khương Đình,Thanh Xuân,"Món Nhật, hải sản, nhà hàng cao cấp",945000,4.20
978,Hatoyama Japanese Restaurant Hoàng - Tô Ngọc Vân,Tây Hồ,"Món Nhật, hải sản, nhà hàng cao cấp",935000,3.90
908,Hatoyama Japanese Restaurant 267 Tạ Quang Bửu,Hai Bà Trưng,"Món Nhật, hải sản, nhà hàng cao cấp",915000,4.40
659,Lẩu Nấm Ashima Hoàng Đạo Thúy,Cầu Giấy,"Lẩu nấm thiên nhiên, Món Á",900000,4.70


## 4. Phân Tích Phân Phối Biến Định Tính (Categorical Analysis)
Thay thế cho **Bar chart** và **Pie chart**, ta sử dụng bảng tần suất và tỷ lệ phần trăm (`%`).

In [11]:
# 1. Phân phối theo Quận / Huyện (Locality)
locality_summary = pd.DataFrame({
    'Số lượng quán': df['Locality'].value_counts(),
    'Tỷ lệ (%)': (df['Locality'].value_counts(normalize=True) * 100).round(2)
})
print('--- Phân phối nhà hàng theo Quận / Huyện ---')
display(locality_summary.head(10))

# 2. Phân phối theo Mức giá (Price range: 1 = Bình dân, 4 = Cao cấp)
price_summary = pd.DataFrame({
    'Số lượng quán': df['Price range'].value_counts().sort_index(),
    'Tỷ lệ (%)': (df['Price range'].value_counts(normalize=True).sort_index() * 100).round(2)
})
print('\n--- Phân phối theo Phân khúc giá (Price range 1 - 4) ---')
display(price_summary)

# 3. Tỷ lệ cung cấp dịch vụ Đặt bàn & Giao hàng
services_summary = pd.DataFrame({
    'Đặt bàn (Table booking)': df['Has Table booking'].value_counts(),
    'Đặt bàn (%)': (df['Has Table booking'].value_counts(normalize=True) * 100).round(2),
    'Giao hàng (Online delivery)': df['Has Online delivery'].value_counts(),
    'Giao hàng (%)': (df['Has Online delivery'].value_counts(normalize=True) * 100).round(2)
})
print('\n--- Dịch vụ Đặt bàn & Giao hàng ---')
display(services_summary)

# 4. Đánh giá chất lượng bằng chữ (Rating text)
rating_text_summary = pd.DataFrame({
    'Số lượng': df['Rating text'].value_counts(),
    'Tỷ lệ (%)': (df['Rating text'].value_counts(normalize=True) * 100).round(2)
})
print('\n--- Phân loại đánh giá (Rating text) ---')
display(rating_text_summary)

# 5. Tách và đếm tần suất Top thể loại ẩm thực (Cuisines)
all_cuisines = df['Cuisines'].dropna().apply(lambda x: [c.strip() for c in str(x).split(',') if c.strip()])
cuisine_series = pd.Series([c for sublist in all_cuisines for c in sublist])
top_cuisines = pd.DataFrame({
    'Số lượng quán phục vụ': cuisine_series.value_counts().head(15),
    'Tỷ lệ trên tổng số quán (%)': (cuisine_series.value_counts().head(15) / len(df) * 100).round(2)
})
print('\n--- Top 15 Thể loại món ăn (Cuisines) phổ biến nhất Hà Nội ---')
display(top_cuisines)

--- Phân phối nhà hàng theo Quận / Huyện ---


,Số lượng quán,Tỷ lệ (%)
Locality,,
Cầu Giấy,138,11.40
Đống Đa,137,11.32
Hoàn Kiếm,133,10.99
Hai Bà Trưng,108,8.93
Ba Đình,100,8.26
Thanh Xuân,92,7.60
Hòa Lạc,90,7.44
Tây Hồ,71,5.87
Nam Từ Liêm,66,5.45



--- Phân phối theo Phân khúc giá (Price range 1 - 4) ---


,Số lượng quán,Tỷ lệ (%)
Price range,,
1,433,35.79
2,292,24.13
3,335,27.69
4,150,12.40



--- Dịch vụ Đặt bàn & Giao hàng ---


,Đặt bàn (Table booking),Đặt bàn (%),Giao hàng (Online delivery),Giao hàng (%)
No,709,58.60,326,26.94
Yes,501,41.40,884,73.06



--- Phân loại đánh giá (Rating text) ---


,Số lượng,Tỷ lệ (%)
Rating text,,
Rất tốt,636,52.56
Tốt,350,28.93
Xuất sắc,212,17.52
Trung bình,12,0.99



--- Top 15 Thể loại món ăn (Cuisines) phổ biến nhất Hà Nội ---


,Số lượng quán phục vụ,Tỷ lệ trên tổng số quán (%)
Món Việt,699,57.77
Lẩu,311,25.70
Đồ nướng,92,7.60
Đồ Âu,81,6.69
quán phở,81,6.69
Món Nhật,77,6.36
hải sản,76,6.28
Buffet,66,5.45
Món Ý,61,5.04
Đồ nướng Hàn Quốc,60,4.96


## 5. Gom Nhóm & So Sánh Đa Chiều (Aggregation, Pivot & Crosstab)
So sánh chi phí và điểm đánh giá giữa các khu vực và các nhóm dịch vụ.

In [12]:
# 1. So sánh chi phí & rating giữa các Quận có từ 20 quán trở lên
top_districts = df['Locality'].value_counts()[lambda x: x >= 20].index
district_comparison = df[df['Locality'].isin(top_districts)].groupby('Locality').agg(
    So_quan=('Restaurant ID', 'count'),
    Chi_phi_TB=('Average Cost for two', 'mean'),
    Chi_phi_Trung_vi=('Average Cost for two', 'median'),
    Rating_TB=('Aggregate rating', 'mean'),
    Rating_Trung_vi=('Aggregate rating', 'median'),
    Votes_TB=('Votes', 'mean')
).sort_values(by='Chi_phi_TB', ascending=False).round(2)

print('--- So sánh chi phí và chất lượng giữa các Quận tiêu biểu ---')
display(district_comparison)

# 2. Pivot Table 2 chiều: Giá trung bình (VNĐ) theo Quận và Xếp loại Rating
pivot_district_rating = pd.pivot_table(
    df[df['Locality'].isin(top_districts)],
    values='Average Cost for two',
    index='Locality',
    columns='Rating text',
    aggfunc='mean'
).round(0)
print('\n--- Giá trung bình cho 2 người (VNĐ) theo Quận và Rating text ---')
display(pivot_district_rating)

# 3. Bảng chéo: Tỷ lệ quán cho phép đặt bàn theo từng phân khúc giá (%)
crosstab_booking_price = pd.crosstab(
    df['Price range'],
    df['Has Table booking'],
    normalize='index'
) * 100
print('\n--- Tỷ lệ dịch vụ Đặt bàn theo Phân khúc giá (%) ---')
display(crosstab_booking_price.round(2))

# 4. Bảng chéo: Tỷ lệ giao hàng trực tuyến theo Phân khúc giá (%)
crosstab_delivery_price = pd.crosstab(
    df['Price range'],
    df['Has Online delivery'],
    normalize='index'
) * 100
print('\n--- Tỷ lệ dịch vụ Giao hàng Online theo Phân khúc giá (%) ---')
display(crosstab_delivery_price.round(2))

--- So sánh chi phí và chất lượng giữa các Quận tiêu biểu ---


,So_quan,Chi_phi_TB,Chi_phi_Trung_vi,Rating_TB,Rating_Trung_vi,Votes_TB
Locality,,,,,,
Tây Hồ,71,"290,746.48","244,000.00",4.11,4.10,595.38
Cầu Giấy,138,"287,557.97","222,500.00",4.21,4.30,584.66
Long Biên,30,"281,566.67","221,500.00",4.18,4.20,665.67
Hai Bà Trưng,108,"281,231.48","235,000.00",4.16,4.20,614.31
Đống Đa,137,"280,897.81","218,000.00",4.15,4.10,662.99
Thanh Xuân,92,"280,391.30","217,500.00",4.18,4.20,644.04
Nam Từ Liêm,66,"272,681.82","217,500.00",4.13,4.10,640.05
Hà Đông,60,"260,900.00","236,000.00",4.08,4.10,521.97
Ba Đình,100,"258,510.00","215,500.00",4.16,4.20,650.84



--- Giá trung bình cho 2 người (VNĐ) theo Quận và Rating text ---


Rating text,Rất tốt,Trung bình,Tốt,Xuất sắc
Locality,,,,
Ba Đình,"249,000.00","96,000.00","192,821.00","382,850.00"
Bắc Từ Liêm,"181,667.00",NaN,"235,625.00","174,000.00"
Cầu Giấy,"305,045.00",NaN,"179,559.00","353,816.00"
Hai Bà Trưng,"288,491.00","95,000.00","184,267.00","415,300.00"
Hoàn Kiếm,"214,500.00",NaN,"196,925.00","361,652.00"
Hoàng Mai,"269,278.00",NaN,"206,083.00","263,200.00"
Hà Đông,"309,649.00","89,000.00","169,611.00","263,750.00"
Hòa Lạc,"234,255.00","82,500.00","140,345.00","287,083.00"
Long Biên,"263,389.00",NaN,"232,286.00","416,000.00"



--- Tỷ lệ dịch vụ Đặt bàn theo Phân khúc giá (%) ---


Has Table booking,No,Yes
Price range,,
1,99.54,0.46
2,68.15,31.85
3,20.00,80.00
4,8.00,92.00



--- Tỷ lệ dịch vụ Giao hàng Online theo Phân khúc giá (%) ---


Has Online delivery,No,Yes
Price range,,
1,17.09,82.91
2,15.07,84.93
3,37.91,62.09
4,54.00,46.00


## 6. Phân Tích Tương Quan & Kiểm Định Thống Kê (Hypothesis Testing)
Thay thế Scatter Plot & Correlation Heatmap bằng hệ số tương quan số học và các kiểm định $p\text{-value}$ chính xác.

In [13]:
# 1. Ma trận tương quan Pearson & Spearman
numeric_cols = ['Average Cost for two', 'Price range', 'Aggregate rating', 'Votes']

print('--- Ma trận hệ số tương quan Pearson (Tuyến tính) ---')
pearson_corr = df[numeric_cols].corr(method='pearson').round(3)
display(pearson_corr)

print('\n--- Ma trận hệ số tương quan Spearman (Đơn điệu / Phi tuyến) ---')
spearman_corr = df[numeric_cols].corr(method='spearman').round(3)
display(spearman_corr)

# 2. Kiểm định T-Test: Quán có Đặt bàn (Table booking = Yes) có điểm Rating cao hơn không?
print('\n' + '='*70)
print('KIỂM ĐỊNH 1: ĐIỂM RATING GIỮA QUÁN CÓ ĐẶT BÀN VS KHÔNG ĐẶT BÀN (T-TEST)')
print('='*70)
booking_yes = df[df['Has Table booking'] == 'Yes']['Aggregate rating'].dropna()
booking_no = df[df['Has Table booking'] == 'No']['Aggregate rating'].dropna()

t_stat, p_val_t = stats.ttest_ind(booking_yes, booking_no, equal_var=False)
print(f'Số quán có đặt bàn: {len(booking_yes)}, Điểm TB: {booking_yes.mean():.2f}')
print(f'Số quán không đặt bàn: {len(booking_no)}, Điểm TB: {booking_no.mean():.2f}')
print(f't-statistic: {t_stat:.4f}, p-value: {p_val_t:.4e}')
if p_val_t < 0.05:
    print('=> KẾT LUẬN: Có sự khác biệt có ý nghĩa thống kê về điểm đánh giá giữa 2 nhóm (p < 0.05).')
    if booking_yes.mean() > booking_no.mean():
        print('   Nhóm nhà hàng có dịch vụ đặt bàn có điểm đánh giá trung bình cao hơn rõ rệt.')
else:
    print('=> KẾT LUẬN: Chưa đủ bằng chứng cho thấy sự khác biệt về điểm đánh giá giữa 2 nhóm (p >= 0.05).')

# 3. Kiểm định ANOVA: Chi phí trung bình có thực sự khác nhau giữa các quận chính?
print('\n' + '='*70)
print('KIỂM ĐỊNH 2: KHÁC BIỆT CHI PHÍ GIỮA CÁC QUẬN TIÊU BIỂU (ONE-WAY ANOVA)')
print('='*70)
district_cost_groups = [
    group['Average Cost for two'].values
    for _, group in df[df['Locality'].isin(top_districts)].groupby('Locality')
]
f_stat, p_val_f = stats.f_oneway(*district_cost_groups)
print(f'F-statistic: {f_stat:.4f}, p-value: {p_val_f:.4e}')
if p_val_f < 0.05:
    print('=> KẾT LUẬN: Có sự khác biệt có ý nghĩa thống kê về chi phí giữa các quận tại Hà Nội (p < 0.05).')
else:
    print('=> KẾT LUẬN: Chi phí giữa các quận không có sự chênh lệch đáng kể.')

# 4. Kiểm định Chi-square: Mối liên hệ giữa Phân khúc giá và Dịch vụ Giao hàng Online
print('\n' + '='*70)
print('KIỂM ĐỊNH 3: PHÂN KHÚC GIÁ VÀ DỊCH VỤ ONLINE DELIVERY (CHI-SQUARE)')
print('='*70)
contingency = pd.crosstab(df['Price range'], df['Has Online delivery'])
chi2_val, p_val_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'chi2: {chi2_val:.4f}, p-value: {p_val_chi:.4e}, Bậc tự do (dof): {dof}')
if p_val_chi < 0.05:
    print('=> KẾT LUẬN: Tỷ lệ cung cấp dịch vụ giao hàng online phụ thuộc chặt chẽ vào phân khúc giá (p < 0.05).')
else:
    print('=> KẾT LUẬN: Hai yếu tố độc lập thống kê với nhau.')

--- Ma trận hệ số tương quan Pearson (Tuyến tính) ---


,Average Cost for two,Price range,Aggregate rating,Votes
Average Cost for two,1.00,0.94,0.31,0.16
Price range,0.94,1.00,0.29,0.16
Aggregate rating,0.31,0.29,1.00,0.20
Votes,0.16,0.16,0.20,1.00



--- Ma trận hệ số tương quan Spearman (Đơn điệu / Phi tuyến) ---


,Average Cost for two,Price range,Aggregate rating,Votes
Average Cost for two,1.00,0.96,0.28,0.22
Price range,0.96,1.00,0.29,0.25
Aggregate rating,0.28,0.29,1.00,0.23
Votes,0.22,0.25,0.23,1.00



KIỂM ĐỊNH 1: ĐIỂM RATING GIỮA QUÁN CÓ ĐẶT BÀN VS KHÔNG ĐẶT BÀN (T-TEST)
Số quán có đặt bàn: 501, Điểm TB: 4.24
Số quán không đặt bàn: 709, Điểm TB: 4.08
t-statistic: 8.8600, p-value: 2.9358e-18
=> KẾT LUẬN: Có sự khác biệt có ý nghĩa thống kê về điểm đánh giá giữa 2 nhóm (p < 0.05).
   Nhóm nhà hàng có dịch vụ đặt bàn có điểm đánh giá trung bình cao hơn rõ rệt.

KIỂM ĐỊNH 2: KHÁC BIỆT CHI PHÍ GIỮA CÁC QUẬN TIÊU BIỂU (ONE-WAY ANOVA)
F-statistic: 2.1515, p-value: 9.7583e-03
=> KẾT LUẬN: Có sự khác biệt có ý nghĩa thống kê về chi phí giữa các quận tại Hà Nội (p < 0.05).

KIỂM ĐỊNH 3: PHÂN KHÚC GIÁ VÀ DỊCH VỤ ONLINE DELIVERY (CHI-SQUARE)
chi2: 118.5349, p-value: 1.5957e-25, Bậc tự do (dof): 3
=> KẾT LUẬN: Tỷ lệ cung cấp dịch vụ giao hàng online phụ thuộc chặt chẽ vào phân khúc giá (p < 0.05).
